In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Module Import

In [ ]:
!pip install pandas ftfy google-generativeai
!pip install torch

In [ ]:
import pandas as pd
import numpy as np
from  matplotlib import pylab as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# Data Import

In [ ]:
dataset = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP & LLMs/Finance News Sentiments (LLM/dataset.csv")

dataset

,sentiment,text
0,positive,"All banks, lending institutions may allow a th..."
1,neutral,"Not so fast, Drake..."
2,positive,FNF - ong 19.43. Trailing Stop 21.04 from 6 ...
3,positive,Dow opens down almost 500 points after China h...
4,positive,U.S. weekly active oil-rig count edges up by 1...
...,...,...
32578,negative,Airbnb to lose £325m in London bookings after ...
32579,positive,Tesla surpasses 2019 goal and delivers 367500 ...
32580,positive,Hundreds of French workers at Caterpillar take...
32581,neutral,Emerging-economy central banks from India to B...


In [ ]:
dataset.dropna(inplace=True)

# Phase 1: Data Preparation & Tokenization

## Step 1 (Updated): Ingesting the CSV

In [ ]:
df = dataset.copy()

# 2. Extract the 'text' column and join it into one large string
# We add a newline character between each headline so the model learns where one ends and another begins.
text = '\n'.join(df['text'].dropna().astype(str).tolist())

print(f"Total characters in dataset: {len(text)}")

# 3. Extract the Vocabulary (same as before, but now much larger!)
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary Size: {vocab_size}")

# 4. Build the Tokenizer
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi.get(c, 0) for c in s] # .get() handles unknown characters safely
decode = lambda l: ''.join([itos.get(i, '') for i in l])

# Encode the entire text
data = torch.tensor(encode(text), dtype=torch.long)

# 5. Batching setup
block_size = 64
batch_size = 32
torch.manual_seed(1337)

def get_batch():
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch()

Total characters in dataset: 3118962
Vocabulary Size: 561


# Phase 2: Building the Core Components

## Step 4: The Mathematical Core — Self-Attention

In [ ]:
n_embd = 128 # Embedding dimension
head_size = 16 # Dimensionality of the attention head

class Head(nn.Module):
    """ One head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        # Linear layers to generate Keys, Queries, and Values
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # 'tril' is the lower triangular mask.
        # We register it as a buffer because it's a fixed matrix, not a learned parameter.
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape # Batch, Time (block_size), Channels (n_embd)

        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)

        # Compute attention scores ("affinities")
        # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
        wei = q @ k.transpose(-2, -1) * (head_size ** -0.5) # The mathematical formula: QK^T / sqrt(d_k)

        # THE MASK: We overwrite the upper triangle with negative infinity.
        # This prevents tokens from "looking into the future" to guess the next character.
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # Softmax turns the affinities into nice probabilities that sum to 1
        wei = F.softmax(wei, dim=-1)

        # Perform the weighted aggregation of the Values
        v = self.value(x) # (B, T, head_size)
        out = wei @ v     # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        return out

# Let's test our Head with the embeddings we created previously
# Note: we need the `token_embedding_table` from the previous step
token_embedding_table = nn.Embedding(vocab_size, n_embd)
embedded_x = token_embedding_table(xb)

attention_head = Head(head_size)
contextualized_output = attention_head(embedded_x)

print(f"Shape after Self-Attention: {contextualized_output.shape}")
# Expected: [4, 8, 16] (Batch, Time, Head Size)

Shape after Self-Attention: torch.Size([32, 64, 16])


## Step 5: Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        # Create a list of our previously defined 'Head' modules
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

        # A final linear layer to project the concatenated results back into the embedding pathway
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        # Run all heads in parallel and concatenate along the channel dimension (dim=-1)
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

# Let's test it! We want 4 heads, each of size 8 (so they concatenate back to our n_embd of 32)
num_heads = 4
head_size = n_embd // num_heads # 32 // 4 = 8

mha = MultiHeadAttention(num_heads, head_size)
mha_output = mha(embedded_x)

print(f"Shape after Multi-Head Attention: {mha_output.shape}")
# Expected: [4, 8, 32] (Batch, Time, n_embd)

Shape after Multi-Head Attention: torch.Size([32, 64, 128])


## Step 6: The Feed-Forward Network (FFN)

In [ ]:
class FeedForward(nn.Module):
    """ A simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            # In the original Transformer paper, the hidden layer is 4x the size of the embedding
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            # A projection layer going back to the original embedding dimension
            nn.Linear(4 * n_embd, n_embd)
        )

    def forward(self, x):
        return self.net(x)

# Test the Feed-Forward Network
ffn = FeedForward(n_embd)
ffn_output = ffn(mha_output)

print(f"Shape after Feed-Forward Network: {ffn_output.shape}")
# Expected: [4, 8, 32] (Batch, Time, n_embd)

Shape after Feed-Forward Network: torch.Size([32, 64, 128])


# Phase 3: Assembly and Training

## Step 7: The Transformer Block

In [ ]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)

        # Layer Normalization
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # x = x + ... is the residual connection!
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## Step 8: The Full Language Model

In [ ]:
class LanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # 1. Token Embedding (from Step 3)
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        # 2. Positional Embedding (so the model understands sequence order)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # 3. Stack of Transformer Blocks
        self.blocks = nn.Sequential(
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4), # 3 layers deep
        )

        # 4. Final Layer Norm and the Output Projection (the "lm_head")
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Get embeddings
        tok_emb = self.token_embedding_table(idx) # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T, n_embd)

        # Combine them
        x = tok_emb + pos_emb

        # Pass through the transformer blocks
        x = self.blocks(x)
        x = self.ln_f(x)

        # Project back to the vocabulary size to get raw scores (logits)
        logits = self.lm_head(x) # (B, T, vocab_size)

        # Calculate Loss if targets are provided
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            # PyTorch's cross_entropy expects 2D inputs, so we flatten the Batch and Time dimensions
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # This function generates new text character by character
        for _ in range(max_new_tokens):
            # Crop context to the block_size so we don't overflow our positional embeddings
            idx_cond = idx[:, -block_size:]

            # Get predictions for the next character
            logits, loss = self(idx_cond)

            # Focus only on the last time step
            logits = logits[:, -1, :]

            # Apply softmax to convert raw scores into probabilities
            probs = F.softmax(logits, dim=-1)

            # Sample from the distribution (adds randomness)
            idx_next = torch.multinomial(probs, num_samples=1)

            # Append the new character to our running sequence
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## Step 9: The Training Loop and Generation

In [ ]:
# 1. Initialize the model
model = LanguageModel()

# 2. Set up the Optimizer (AdamW is standard for Transformers)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print("Starting training...")
steps = 5000 # Increase this for better results!

# 3. The Training Loop
for step in range(steps):
    # Grab a batch of data from your CSV
    xb, yb = get_batch()

    # Forward pass: evaluate the loss
    logits, loss = model(xb, yb)

    # Backward pass: calculate gradients and update weights
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    # Print progress every 200 steps
    if step % 200 == 0:
        print(f"Step {step} | Loss: {loss.item():.4f}")

print(f"Final Loss: {loss.item():.4f}")

# 4. Generate Text!
print("\n--- Generating Text ---")
# Kick it off with a starting token (e.g., a newline character: index 0)
context = torch.zeros((1, 1), dtype=torch.long)
generated_tokens = model.generate(context, max_new_tokens=200)[0].tolist()

print(decode(generated_tokens))

Starting training...
Step 0 | Loss: 6.5138
Step 200 | Loss: 2.6956
Step 400 | Loss: 2.4972
Step 600 | Loss: 2.3011
Step 800 | Loss: 2.2499
Step 1000 | Loss: 2.0414
Step 1200 | Loss: 2.1623
Step 1400 | Loss: 1.9231
Step 1600 | Loss: 1.8843
Step 1800 | Loss: 1.9309
Step 2000 | Loss: 1.8240
Step 2200 | Loss: 1.7635
Step 2400 | Loss: 1.7886
Step 2600 | Loss: 1.7276
Step 2800 | Loss: 1.7497
Step 3000 | Loss: 1.7493
Step 3200 | Loss: 1.9246
Step 3400 | Loss: 1.6250
Step 3600 | Loss: 1.7257
Step 3800 | Loss: 1.7741
Step 4000 | Loss: 1.8261
Step 4200 | Loss: 1.7047
Step 4400 | Loss: 1.9095
Step 4600 | Loss: 1.7147
Step 4800 | Loss: 1.7679
Final Loss: 1.6084

--- Generating Text ---

OD2ress Global Tax Lithern Studitis Wootoful?viadtingson Ammắoss Relating Turnises Amithings Corp...
Finnish in stores 13% profit candidal potential speechnical dowthp attackdown and leaven its vift i
